#### Feature Engineering

##### Importing Libraries and Loading Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
project_root = Path(".").resolve().parent
df = pd.read_parquet(project_root / 'data' / 'processed' / "london_housing_v2.parquet")

In [3]:
pc = pd.read_csv(project_root/'data'/'supplementary'/'london_postcodes.csv')

##### Merge and Features

In [4]:
pc.columns

Index(['Postcode', 'In Use?', 'Latitude', 'Longitude', 'Easting', 'Northing',
       'Grid Ref', 'County', 'District', 'Ward', 'District Code', 'Ward Code',
       'Country', 'County Code', 'Introduced', 'Terminated', 'Parish',
       'National Park', 'Population', 'Households', 'Built up area',
       'Lower layer super output area', 'Rural/urban', 'Region', 'Altitude',
       'London zone', 'LSOA Code', 'Local authority', 'MSOA Code',
       'Middle layer super output area', 'Parish Code', 'Census output area',
       'Index of Multiple Deprivation', 'Quality', 'User Type', 'Last updated',
       'Nearest station', 'Distance to station', 'Postcode area',
       'Postcode district', 'Police force', 'Plus Code', 'Average Income',
       'Travel To Work Area', 'ITL level 2', 'ITL level 3', 'UPRNs',
       'Distance to sea', 'LSOA21 Code', 'Lower layer super output area 2021',
       'MSOA21 Code', 'Middle layer super output area 2021',
       'Census output area 2021', 'IMD decile', 'Co

In [5]:
pc = pc[['Postcode', 'Latitude', 'Longitude', 'London zone', 'IMD decile',
         'Index of Multiple Deprivation', 'Average Income', 'Distance to station']]
pc.columns = ['postcode', 'lat', 'lon', 'zone', 'imd_decile',
              'imd_index', 'avg_income', 'dist_station']

In [6]:
df = df.merge(pc, how='left', on='postcode')

In [11]:
df['log_income'] = np.log10(df['avg_income'])

In [14]:
df['postcode_district'] = df["postcode"].str.split(' ').str[0]

In [15]:
df.columns

Index(['priceper', 'year', 'dateoftransfer', 'propertytype', 'duration',
       'price', 'postcode', 'lad23cd', 'transactionid', 'lmk_key', 'tfarea',
       'numberrooms', 'classt', 'CURRENT_ENERGY_EFFICIENCY',
       'POTENTIAL_ENERGY_EFFICIENCY', 'CONSTRUCTION_AGE_BAND', 'month',
       'log_price', 'borough', 'log_tfarea', 'age_year', 'lat', 'lon', 'zone',
       'imd_decile', 'imd_index', 'avg_income', 'dist_station', 'log_income',
       'postcode_district'],
      dtype='object')

In [16]:
df = pd.get_dummies(df, columns=['propertytype', 'duration'],
                    dtype=int)

In [18]:
df = df.rename(columns={
    'CURRENT_ENERGY_EFFICIENCY': 'energy_current',
    'POTENTIAL_ENERGY_EFFICIENCY': 'energy_potential',
})

In [20]:
features = ['log_tfarea', 'numberrooms', 'age_year',
    'energy_current', 'energy_potential',
    'lat', 'lon', 'zone', 'imd_index', 'log_income', 'dist_station',
    'propertytype_D', 'propertytype_F', 'propertytype_S', 'propertytype_T',
    'duration_F', 'duration_L', 'duration_U',
    'year', 'month']

keep = ['price', 'log_price', 'postcode_district', 'lad23cd'] + features

In [21]:
out = df[keep].copy()

In [22]:
out_path = project_root/'data' /'processed' /'london_features.parquet'
out.to_parquet(out_path, index=False)